# Public Transport Ridership Analysis: Exploratory Data Analysis (EDA)
**End-to-End Analysis with Python, Pandas, Matplotlib, and Excel/CSV Processing**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github)

This notebook provides statistical evaluations, hourly/daily transit trends, peak vs. off-peak volume comparisons, route performance benchmarks, and multi-sheet Excel report exports.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Set high-DPI dark aesthetic for charts
plt.style.use('dark_background')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 140

# Resolve CSV path for both local execution and Google Colab
csv_path = '../data/ridership.csv' if os.path.exists('../data/ridership.csv') else 'data/ridership.csv'
if not os.path.exists(csv_path):
    print('Generating dataset on the fly...')
    # Fallback inline generation if running standalone in Colab
    from datetime import datetime, timedelta
    np.random.seed(42)
    n = 10000
    start = datetime(2026, 1, 1)
    secs = np.random.randint(0, 365*24*3600, n)
    secs.sort()
    ts = [start + timedelta(seconds=int(s)) for s in secs]
    r_ids = [f'R-{101 + i}' for i in range(10)]
    s_ids = [f'S-{i:02d}' for i in range(1, 21)]
    routes = np.random.choice(r_ids, n)
    stations = np.random.choice(s_ids, n)
    peaks = [1 if (t.weekday() < 5 and (7 <= t.hour <= 9 or 17 <= t.hour <= 19)) else 0 for t in ts]
    passengers = [int(35 * (2.5 if p else 1.2) + np.random.normal(0, 8)) for p in peaks]
    df = pd.DataFrame({'timestamp': [t.strftime('%Y-%m-%d %H:%M:%S') for t in ts], 'route_id': routes, 'station_id': stations, 'passenger_count': np.maximum(5, passengers), 'is_peak': peaks})
    os.makedirs('data', exist_ok=True)
    df.to_csv('data/ridership.csv', index=False)
    csv_path = 'data/ridership.csv'

df = pd.read_csv(csv_path)
df['datetime'] = pd.to_datetime(df['timestamp'])
df['date'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour
df['day_name'] = df['datetime'].dt.day_name()
print(f'Successfully loaded dataset: {len(df):,} rows and {len(df.columns)} columns.')
df.head()

## 1. High-Level Dataset Summary & Statistical Overview

In [ ]:
total_passengers = df['passenger_count'].sum()
total_trips = len(df)
avg_riders = df['passenger_count'].mean()
peak_riders = df[df['is_peak'] == 1]['passenger_count'].sum()
offpeak_riders = df[df['is_peak'] == 0]['passenger_count'].sum()

print('================ PUBLIC TRANSPORT KPI SUMMARY ================')
print(f'Total Passenger Volume:      {total_passengers:,}')
print(f'Total Logged Trips:          {total_trips:,}')
print(f'Average Volume per Trip:     {avg_riders:.2f} passengers')
print(f'Peak Passenger Count:        {peak_riders:,} ({peak_riders/total_passengers*100:.1f}%)')
print(f'Off-Peak Passenger Count:    {offpeak_riders:,} ({offpeak_riders/total_passengers*100:.1f}%)')
print('==============================================================')
df[['passenger_count', 'is_peak']].describe()

## 2. Peak vs. Off-Peak Comparative Analysis

In [ ]:
peak_stats = df.groupby('is_peak').agg(
    total_volume=('passenger_count', 'sum'),
    avg_per_trip=('passenger_count', 'mean'),
    std_dev=('passenger_count', 'std'),
    trip_count=('passenger_count', 'count')
).reset_index()
peak_stats['period'] = peak_stats['is_peak'].map({1: 'Peak Hours', 0: 'Off-Peak Hours'})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0B0F19')
for ax in (ax1, ax2):
    ax.set_facecolor('#111827')
    ax.grid(True, linestyle='--', alpha=0.3, color='#374151')

ax1.bar(peak_stats['period'], peak_stats['total_volume'], color=['#10B981', '#F59E0B'], width=0.5)
ax1.set_title('Cumulative Passenger Volume (Total)', fontsize=12, fontweight='bold', color='#F3F4F6')
ax1.set_ylabel('Passengers', color='#9CA3AF')
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'{int(x):,}'))

ax2.bar(peak_stats['period'], peak_stats['avg_per_trip'], color=['#3B82F6', '#8B5CF6'], width=0.5)
ax2.set_title('Average Passenger Density Per Vehicle Trip', fontsize=12, fontweight='bold', color='#F3F4F6')
ax2.set_ylabel('Average Passengers / Trip', color='#9CA3AF')

plt.tight_layout()
plt.show()

## 3. Hourly Transit Dynamics & Peak Hour Detection

In [ ]:
hourly_trend = df.groupby('hour')['passenger_count'].agg(['sum', 'mean']).reset_index()

plt.figure(figsize=(14, 6))
plt.gca().set_facecolor('#111827')
plt.gcf().patch.set_facecolor('#0B0F19')

plt.plot(hourly_trend['hour'], hourly_trend['sum'], marker='o', linewidth=2.5, color='#38BDF8', label='Hourly Total Volume')
plt.fill_between(hourly_trend['hour'], hourly_trend['sum'], color='#38BDF8', alpha=0.18)

# Highlight Peak Windows
plt.axvspan(7, 9.9, color='#F59E0B', alpha=0.2, label='Morning Peak (07:00 - 10:00)')
plt.axvspan(17, 19.9, color='#EF4444', alpha=0.2, label='Evening Peak (17:00 - 20:00)')

plt.title('24-Hour Passenger Traffic Profile across Transport Network', fontsize=14, fontweight='bold', color='#F3F4F6')
plt.xlabel('Hour of Day (00:00 - 23:00)', fontsize=11, color='#9CA3AF')
plt.ylabel('Total Passengers Recorded', fontsize=11, color='#9CA3AF')
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.3, color='#374151')
plt.legend(facecolor='#1F2937', edgecolor='#374151')
plt.tight_layout()
plt.show()

## 4. Route Performance & Station Ranking

In [ ]:
top_routes = df.groupby('route_id')['passenger_count'].sum().sort_values(ascending=False).head(5)
top_stations = df.groupby('station_id')['passenger_count'].sum().sort_values(ascending=False).head(5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0B0F19')
for ax in (ax1, ax2):
    ax.set_facecolor('#111827')
    ax.grid(True, linestyle='--', alpha=0.3, color='#374151', axis='x')

ax1.barh(top_routes.index[::-1], top_routes.values[::-1], color='#6366F1', height=0.55)
ax1.set_title('Top 5 High-Demand Transit Routes', fontsize=12, fontweight='bold', color='#F3F4F6')
ax1.set_xlabel('Passenger Volume', color='#9CA3AF')

ax2.barh(top_stations.index[::-1], top_stations.values[::-1], color='#EC4899', height=0.55)
ax2.set_title('Top 5 High-Demand Passenger Stations', fontsize=12, fontweight='bold', color='#F3F4F6')
ax2.set_xlabel('Passenger Volume', color='#9CA3AF')

plt.tight_layout()
plt.show()

## 5. Day of Week Patterns & Weekly Distribution

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_pivot = df.groupby(['day_name', 'is_peak'])['passenger_count'].sum().unstack(fill_value=0).reindex(day_order)

ax = dow_pivot.plot(kind='bar', stacked=True, figsize=(12, 5), color=['#10B981', '#F59E0B'])
plt.gcf().patch.set_facecolor('#0B0F19')
ax.set_facecolor('#111827')
plt.title('Weekly Ridership Distribution (Peak vs Off-Peak by Day)', fontsize=13, fontweight='bold', color='#F3F4F6')
plt.ylabel('Total Passengers', color='#9CA3AF')
plt.xlabel('Day of Week', color='#9CA3AF')
plt.xticks(rotation=0, color='#D1D5DB')
plt.grid(True, linestyle='--', alpha=0.3, color='#374151', axis='y')
plt.legend(['Off-Peak', 'Peak'], facecolor='#1F2937', edgecolor='#374151')
plt.tight_layout()
plt.show()

## 6. Multi-Sheet Excel Summary Export

In [ ]:
export_path = 'data/ridership_report_colab.xlsx'
with pd.ExcelWriter(export_path, engine='openpyxl') as writer:
    top_routes.to_frame('volume').to_excel(writer, sheet_name='Top_Routes')
    top_stations.to_frame('volume').to_excel(writer, sheet_name='Top_Stations')
    dow_pivot.to_excel(writer, sheet_name='Day_Of_Week_Analysis')
    hourly_trend.to_excel(writer, sheet_name='Hourly_Trends', index=False)

print(f'Exported comprehensive Excel workbook to {export_path}')